# Análise e limpeza do INCT

In [1]:
!pip install pandas


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: pip install --upgrade pip


In [ ]:
import json
from pathlib import Path

import pandas as pd

# Pasta raiz do repositório (funciona rodando de notebooks/ ou da raiz)
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ORIGINAIS = RAIZ / "dados" / "originais"

with open(ORIGINAIS / "INCT" / "data.json", encoding="utf-8") as f:
    raw = json.load(f)

df = pd.DataFrame([
    {"nivel_m": float(nivel.removesuffix("m").replace("_", ".")),
     "bairro": bairro, **valores}
    for nivel, bairros in raw.items()
    for bairro, valores in bairros.items()
])
with pd.option_context("display.max_rows", None):
    display(df) 


## Dica 1 — Prédios afetados devem crescer (ou ficar iguais) quando o rio sobe

**Regra esperada:** para um mesmo bairro, se o nível do Guaíba sobe, a água cobre tudo o que já cobria **e mais**.
Logo, `affected_buildings` deve ser **monotônico não-decrescente** em relação a `nivel_m`.

Qualquer caso em que o valor **diminui** de um nível para o seguinte é uma inconsistência.

In [15]:
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

# Ordena por bairro e nível (não confiar na ordem do arquivo — ver dica 3)
df = df.sort_values(["bairro", "nivel_m"]).reset_index(drop=True)
# Diferença em relação ao nível anterior do mesmo bairro
df["anterior"] = df.groupby("bairro")["affected_buildings"].shift(1)
df["diff"] = df["affected_buildings"] - df["anterior"]

quedas = df[df["diff"] < 0][["bairro", "nivel_m", "anterior", "affected_buildings", "diff"]]
print(f"{len(quedas)} quedas encontradas em {quedas['bairro'].nunique()} bairros")
quedas

12 quedas encontradas em 5 bairros


,bairro,nivel_m,anterior,affected_buildings,diff
214,Arquipélago,3.1,4685.0,4680,-5.0
219,Arquipélago,3.6,4685.0,4680,-5.0
224,Arquipélago,4.1,4685.0,4674,-11.0
229,Arquipélago,4.6,4685.0,4680,-5.0
234,Arquipélago,5.1,4685.0,4680,-5.0
237,Arquipélago,5.4,4688.0,4680,-8.0
1274,Centro Histórico,9.7,2050.0,0,-2050.0
1779,Espírito Santo,3.4,637.0,633,-4.0
1780,Espírito Santo,3.5,633.0,25,-608.0
1793,Espírito Santo,4.8,2101.0,47,-2054.0


In [23]:
dc = df.copy()
dc["affected_buildings_original"] = dc["affected_buildings"]
dc["tratamento_d1"] = ""
g = dc.groupby("bairro")["affected_buildings"]
ab = dc["affected_buildings"].astype(float)
viz_ant, viz_pos = g.shift(1), g.shift(-1)

# (a) Zeros isolados: 0 depois de o bairro já ter tido prédios afetados
ja_afetado = g.cummax().groupby(dc["bairro"]).shift(1) > 0
m_zero = (dc["affected_buildings"] == 0) & ja_afetado

# (b) Picos isolados: valor muito maior que os dois vizinhos
m_pico = dc["affected_buildings"] > 3 * np.maximum(viz_ant, viz_pos)

# (c) Patamar inicial alto: todos os níveis antes de uma queda > 50%
#     (desconsiderando as quedas que só existem por causa de um pico já marcado)
queda_grande = (dc["affected_buildings"] < 0.5 * viz_ant) & ~m_pico.groupby(dc["bairro"]).shift(1, fill_value=False) & ~m_zero
depois_da_queda = queda_grande.astype(int).groupby(dc["bairro"]).cumsum() > 0
tem_queda = queda_grande.groupby(dc["bairro"]).transform("any")
m_patamar = tem_queda & ~depois_da_queda

# (d) Ruído pequeno: bairros que AINDA têm quedas depois de remover (a)-(c);
#     neles, valores que se desviam do valor predominante (moda) do bairro
#     em até 1% do total de prédios
restante = dc["affected_buildings"].where(~(m_zero | m_pico | m_patamar))
cai_restante = restante.groupby(dc["bairro"]).transform(lambda s: (s.dropna().diff() < 0).any())
ainda_cai = cai_restante.astype(bool)
moda = g.transform(lambda s: s.mode().iloc[0])
m_ruido = (ainda_cai & (dc["affected_buildings"] != moda)
           & ((dc["affected_buildings"] - moda).abs() <= 0.01 * dc["total_buildings"])
           & ~(m_zero | m_pico | m_patamar))

for mask, rotulo in [(m_zero, "interpolado (zero isolado)"),
                     (m_pico, "interpolado (pico isolado)"),
                     (m_patamar, "interpolado (patamar inicial inconsistente)"),
                     (m_ruido, "interpolado (ruído)")]:
    dc.loc[mask, "tratamento_d1"] = rotulo
    ab[mask] = np.nan

# Interpolação linear pelo nível, dentro de cada bairro.
# limit_direction="both" preenche também o início da série (repete o primeiro valor válido).
ab = ab.groupby(dc["bairro"]).transform(
    lambda s: s.set_axis(dc.loc[s.index, "nivel_m"])
               .interpolate(method="index", limit_direction="both").values)
dc["affected_buildings"] = ab.round().astype(int)

# Recalcula os campos derivados com a mesma regra do INCT (a dica 2 discute essa regra)
dc["percentage"] = np.ceil(dc["affected_buildings"] / dc["total_buildings"] * 100).astype(int)
dc["affected_population"] = np.ceil(dc["population"] * dc["percentage"] / 100).astype(int)

alteradas = dc[dc["tratamento_d1"] != ""]
print(alteradas["tratamento_d1"].value_counts())
alteradas[["bairro", "nivel_m", "affected_buildings_original", "affected_buildings", "tratamento_d1"]]

tratamento_d1
interpolado (ruído)                            7
interpolado (patamar inicial inconsistente)    5
interpolado (zero isolado)                     4
interpolado (pico isolado)                     1
Name: count, dtype: int64


,bairro,nivel_m,affected_buildings_original,affected_buildings,tratamento_d1
213,Arquipélago,3.0,4685,4680,interpolado (ruído)
218,Arquipélago,3.5,4685,4680,interpolado (ruído)
223,Arquipélago,4.0,4685,4680,interpolado (ruído)
224,Arquipélago,4.1,4674,4680,interpolado (ruído)
228,Arquipélago,4.5,4685,4680,interpolado (ruído)
233,Arquipélago,5.0,4685,4680,interpolado (ruído)
236,Arquipélago,5.3,4688,4680,interpolado (ruído)
1274,Centro Histórico,9.7,0,2077,interpolado (zero isolado)
1275,Centro Histórico,9.8,0,2103,interpolado (zero isolado)
1775,Espírito Santo,3.0,637,25,interpolado (patamar inicial inconsistente)


### Verificação: não pode sobrar nenhuma queda

In [24]:
bairros_prob = sorted(quedas["bairro"].unique())
d2 = dc.groupby("bairro")["affected_buildings"].diff()
print("Quedas restantes:", int((d2 < 0).sum()))

fig = px.line(dc[dc["bairro"].isin(bairros_prob)], x="nivel_m", y="affected_buildings",
              facet_col="bairro", facet_col_wrap=3, markers=True,
              category_orders={"bairro": sorted(bairros_prob)},
              labels={"nivel_m": "Nível (m)", "affected_buildings": "Prédios afetados"},
              title="Depois do tratamento (cinza tracejado = original)", height=650)
orig = px.line(df[df["bairro"].isin(bairros_prob)], x="nivel_m", y="affected_buildings",
               facet_col="bairro", facet_col_wrap=3, category_orders={"bairro": sorted(bairros_prob)})
for t in orig.data:
    t.update(line=dict(color="lightgray", dash="dash"), showlegend=False)
    fig.add_trace(t)
fig.update_yaxes(matches=None, showticklabels=True)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.show()

Quedas restantes: 0


## Dica 2 — `percentage` e `affected_population` devem ser coerentes com os outros campos

**Regra esperada:**
* `percentage` = `affected_buildings / total_buildings × 100`
* `affected_population` = `population × percentage / 100`

Qualquer diferença além do arredondamento normal (±0,5) é uma inconsistência.

In [25]:
# Valores esperados a partir dos outros campos (dados originais)
df["pct_esperado"] = df["affected_buildings"] / df["total_buildings"] * 100
df["pop_esperada"] = df["population"] * df["percentage"] / 100      # usa a percentage do próprio arquivo
df["erro_pct"] = df["percentage"] - df["pct_esperado"]
df["erro_pop"] = df["affected_population"] - df["pop_esperada"]

inc_pct = df[df["erro_pct"].abs() > 0.5]
inc_pop = df[df["erro_pop"].abs() > 0.5]
cem_falso = df[(df["percentage"] == 100) & (df["affected_buildings"] < df["total_buildings"])]

print(f"percentage incoerente:          {len(inc_pct)} linhas ({len(inc_pct)/len(df):.0%}) | sempre para cima: {(inc_pct['erro_pct'] > 0).all()}")
print(f"affected_population incoerente: {len(inc_pop)} linhas ({len(inc_pop)/len(df):.0%}) | sempre para cima: {(inc_pop['erro_pop'] > 0).all()}")
print(f"100% sem todos os prédios afetados: {len(cem_falso)} linhas em {sorted(cem_falso['bairro'].unique())}")

# As duas colunas são arredondadas sempre para cima (ceil)?
print("percentage == ceil(pct_esperado):          ", (np.ceil(df["pct_esperado"]) == df["percentage"]).mean() == 1)
print("affected_population == ceil(pop_esperada): ", (np.ceil(df["pop_esperada"] - 1e-9) == df["affected_population"]).mean() == 1)

cols = ["bairro", "nivel_m", "affected_buildings", "total_buildings", "pct_esperado", "percentage",
        "population", "pop_esperada", "affected_population", "erro_pct"]
inc_pct.sort_values("erro_pct", ascending=False)[cols].head(10).round(2)

percentage incoerente:          1294 linhas (19%) | sempre para cima: True
affected_population incoerente: 1314 linhas (20%) | sempre para cima: True
100% sem todos os prédios afetados: 128 linhas em ['Arquipélago', 'Cidade Baixa', 'Farrapos', 'Floresta', 'Praia de Belas', 'Santa Maria Goretti', 'São Geraldo']
percentage == ceil(pct_esperado):           True
affected_population == ceil(pop_esperada):  True


,bairro,nivel_m,affected_buildings,total_buildings,pct_esperado,percentage,population,pop_esperada,affected_population,erro_pct
753,Boa Vista do Sul,7.3,686,2450,28.00,29,2703,783.87,784,1.00
752,Boa Vista do Sul,7.2,686,2450,28.00,29,2703,783.87,784,1.00
13,Aberta dos Morros,4.3,94,4696,2.00,3,9854,295.62,296,1.00
1337,Chapéu do Sol,8.9,935,1798,52.00,53,5547,2939.91,2940,1.00
6229,Vila Assunção,8.2,679,3086,22.00,23,3974,914.02,915,1.00
2074,Floresta,4.5,1791,3731,48.00,49,8798,4311.02,4312,1.00
2057,Farroupilha,9.9,84,227,37.00,38,774,294.12,295,1.00
2056,Farroupilha,9.8,84,227,37.00,38,774,294.12,295,1.00
4529,Pedra Redonda,8.6,152,447,34.00,35,570,199.50,200,1.00
1466,Cidade Baixa,7.6,2152,2339,92.01,93,13014,12103.02,12104,0.99


In [28]:
# Tratamento: recalcular a partir de affected_buildings (já corrigido na dica 1)
dc["percentage_inct"] = df["percentage"]
dc["affected_population_inct"] = df["affected_population"]

dc["percentage"] = (dc["affected_buildings"] / dc["total_buildings"] * 100).round(2)
dc["affected_population"] = (dc["population"] * dc["percentage"] / 100).round().astype(int)

dc["tratamento_d2"] = np.where(
    (dc["percentage"] - dc["percentage_inct"]).abs() > 0.5, "recalculado (percentage)",
    np.where(dc["affected_population"] != dc["affected_population_inct"], "recalculado (arredondamento)", ""))

print(dc["tratamento_d2"].value_counts())


tratamento_d2
                                4032
recalculado (arredondamento)    1351
recalculado (percentage)        1291
Name: count, dtype: int64


### Verificação: os campos derivados devem bater com os outros campos

In [29]:
pct_esperado = (dc["affected_buildings"] / dc["total_buildings"] * 100).round(2)
pop_esperada = dc["population"] * dc["percentage"] / 100
print("percentage incoerente:         ", int(((dc["percentage"] - pct_esperado).abs() > 1e-9).sum()))
print("affected_population incoerente:", int(((dc["affected_population"] - pop_esperada).abs() > 0.5).sum()))
print("100% sem todos afetados:       ", int(((dc["percentage"] == 100) & (dc["affected_buildings"] < dc["total_buildings"])).sum()))

# Total de pessoas afetadas por nível: valores do INCT x recalculados
tot = (dc.groupby("nivel_m")[["affected_population_inct", "affected_population"]].sum()
         .rename(columns={"affected_population_inct": "INCT (original)",
                          "affected_population": "Recalculado"})
         .reset_index().melt(id_vars="nivel_m", var_name="série", value_name="pessoas afetadas"))
fig = px.line(tot, x="nivel_m", y="pessoas afetadas", color="série", markers=True,
              color_discrete_map={"Recalculado": "#1f77b4", "INCT (original)": "lightgray"},
              labels={"nivel_m": "Nível do Guaíba (m)"},
              title="População afetada em Porto Alegre por nível")
fig.update_traces(selector=dict(name="INCT (original)"), line=dict(dash="dash"))
fig.show()

percentage incoerente:          0
affected_population incoerente: 0
100% sem todos afetados:        0


In [30]:
# Todos os níveis têm informação de todos os bairros?
todos_bairros = set(df["bairro"])
todos_niveis = sorted(df["nivel_m"].unique())
print(f"{len(todos_niveis)} níveis ({todos_niveis[0]} a {todos_niveis[-1]} m) e {len(todos_bairros)} bairros distintos")

# Combinações nível x bairro que deveriam existir e não existem
grade = pd.MultiIndex.from_product([todos_niveis, sorted(todos_bairros)], names=["nivel_m", "bairro"])
presentes = pd.MultiIndex.from_frame(df[["nivel_m", "bairro"]])
faltando = grade.difference(presentes).to_frame(index=False)

# Duplicatas (mesmo bairro duas vezes no mesmo nível)
duplicadas = df[df.duplicated(["nivel_m", "bairro"], keep=False)]

# Níveis faltando no intervalo (passo esperado de 0,1 m)
esperados = np.round(np.arange(todos_niveis[0], todos_niveis[-1] + 0.05, 0.1), 1)
niveis_faltando = sorted(set(esperados) - set(np.round(todos_niveis, 1)))

# Campos vazios
nulos = df[["affected_buildings", "total_buildings", "percentage", "population", "affected_population"]].isna().sum()

print("Bairros por nível:", df.groupby("nivel_m")["bairro"].nunique().value_counts().to_dict())
print("Combinações nível x bairro faltando:", len(faltando))
print("Linhas duplicadas:", len(duplicadas))
print("Níveis faltando no intervalo:", niveis_faltando or "nenhum")
print("Valores nulos:", int(nulos.sum()))

if len(faltando):
    display(faltando.groupby("nivel_m")["bairro"].apply(list).to_frame("bairros ausentes"))

71 níveis (3.0 a 10.0 m) e 94 bairros distintos
Bairros por nível: {94: 71}
Combinações nível x bairro faltando: 0
Linhas duplicadas: 0
Níveis faltando no intervalo: nenhum
Valores nulos: 0


# Análise do OSM

## Tabela de escolas e unidades de saúde

In [31]:
with open(RAIZ / "dados" / "originais" / "osm" / "osm_escolas_saude_poa.json", encoding="utf-8") as f:
    osm = pd.json_normalize(json.load(f)["elements"])

# Ways/relations não têm lat/lon próprios: usa o centro calculado pelo Overpass
osm["lat"] = osm["lat"].fillna(osm["center.lat"])
osm["lon"] = osm["lon"].fillna(osm["center.lon"])

colunas = ["type", "id", "tags.amenity", "tags.name", "tags.addr:street",
           "tags.addr:suburb", "lat", "lon"]
colunas = [c for c in colunas if c in osm.columns]
print(f"{len(osm)} pontos")
display(osm["tags.amenity"].value_counts().to_frame("quantidade"))
osm[colunas]

708 pontos


,quantidade
tags.amenity,
school,438
clinic,98
kindergarten,87
hospital,52
doctors,33


,type,id,tags.amenity,tags.name,tags.addr:street,tags.addr:suburb,lat,lon
0,node,476647429,hospital,Unidade Álvaro Alvim - HCPA,Rua Professor Álvaro Alvim,Rio Branco,-30.037137,-51.198245
1,node,598724565,clinic,Estratégia Saúde da Família Tijuca,NaN,NaN,-30.042889,-51.126231
2,node,1118102920,school,Colégio Marista Ipanema,Coronel Marcos,NaN,-30.130579,-51.239623
3,node,1139635710,kindergarten,Escola Moinho,Avenida Soledade,NaN,-30.034895,-51.180255
4,node,1139971906,kindergarten,Creche Berçário e recreação Baby House,Avenida Doutor Nilo Peçanha,NaN,-30.029053,-51.178558
...,...,...,...,...,...,...,...,...
703,relation,6344229,school,Escola Cristóvão Colombo,Rua Rezende Costa,NaN,-29.983076,-51.131544
704,relation,6346576,clinic,Posto de Saúde,Beco José Paris,Sarandi,-29.988678,-51.116854
705,relation,6354469,clinic,Unidade de Saúde SESC,Rua Ernesto Pellanda,Bairro Vila Jardim,-30.035936,-51.148589
706,relation,8397517,school,Colégio Leonardo da Vinci - Unidade Beta,Avenida Icaraí,Cristal,-30.094842,-51.245732


## Mapa

In [33]:
import plotly.express as px

px.scatter_map(osm, lat="lat", lon="lon", color="tags.amenity", hover_name="tags.name",
               zoom=10, height=600, map_style="carto-positron")

## Pontos com nome repetido

In [37]:
# Compara nomes sem diferenciar maiúsculas/minúsculas nem espaços extras
osm["nome_norm"] = osm["tags.name"].str.strip().str.lower()
repetidos = (osm[osm["nome_norm"].notna() & osm.duplicated("nome_norm", keep=False)]
             .sort_values(["nome_norm", "tags.amenity"]))
print(f"{repetidos['nome_norm'].nunique()} nomes repetidos em {len(repetidos)} pontos")
with pd.option_context("display.max_rows", None):
    display(repetidos[colunas])

12 nomes repetidos em 27 pontos


,type,id,tags.amenity,tags.name,tags.addr:street,tags.addr:suburb,lat,lon
472,way,419339793,clinic,Centro Clínico Gaúcho,Avenida Alberto Bins,NaN,-30.028066,-51.218551
224,way,239664213,hospital,Centro Clínico Gaúcho,Avenida do Forte,NaN,-30.012187,-51.153793
123,node,11030466129,school,Colégio Sinodal do Salvador,NaN,NaN,-30.013463,-51.147306
446,way,367324147,school,Colégio Sinodal do Salvador,Avenida Mathilde Trein Renner,NaN,-30.013599,-51.147116
513,way,425196135,kindergarten,Creche Comunitária Tia Ivete,NaN,NaN,-30.048789,-51.150592
514,way,425196740,kindergarten,Creche Comunitária Tia Ivete,NaN,NaN,-30.045210,-51.148295
204,way,235379350,school,Escola Afonso Guerreiro Lima,Rua Banco Inglês,NaN,-30.071847,-51.221474
236,way,243118489,school,Escola Afonso Guerreiro Lima,Rua Guaíba,NaN,-30.110979,-51.112808
276,way,244250676,school,Escola América,Rua Octacílio José Dias,NaN,-30.015383,-51.121649
378,way,317875313,school,Escola América,Rua Padre Ângelo Costa,NaN,-30.078842,-51.167755


In [36]:
# Mapa só com os pontos de nome repetido (cor = nome, símbolo = tipo OSM)
px.scatter_map(repetidos, lat="lat", lon="lon", color="tags.name",
               hover_name="tags.name", hover_data=["type", "id", "tags.amenity"],
               zoom=10, height=600, map_style="carto-positron")

## Análise de duplicatas

Após filtrar os nomes repetidos, exibimos estes pontos no mapa cima e analisamos manualmente quais pontos representam lugares diferentes e quais são redundâncias de um mesmo local.

Duplicatas:
- Unimed: ID's 4117912914 e 420868536
- Escola Visconde do Rio Grande: ID 243463347	
- Escola Saint Hilaire: ID 428647547
- Escola Estadual de 1° Grau Professora Dinah Néri Pereira: ID 6238095094
- Colégio Sinodal do Salvador: ID 11030466129

## Pontos sem nome

In [39]:
sem_nome = osm[osm["tags.name"].isna()]
print(f"{len(sem_nome)} pontos sem nome")
display(sem_nome["tags.amenity"].value_counts().to_frame("quantidade"))
with pd.option_context("display.max_rows", None):
    display(sem_nome[colunas])

24 pontos sem nome


,quantidade
tags.amenity,
school,12
clinic,5
kindergarten,3
hospital,3
doctors,1


,type,id,tags.amenity,tags.name,tags.addr:street,tags.addr:suburb,lat,lon
34,node,3428745569,kindergarten,NaN,NaN,NaN,-30.033714,-51.234507
119,node,8730543929,clinic,NaN,NaN,NaN,-30.043274,-51.153660
120,node,8730543930,clinic,NaN,NaN,NaN,-30.043276,-51.153849
218,way,237709752,hospital,NaN,NaN,NaN,-30.015223,-51.158118
275,way,244050597,school,NaN,NaN,NaN,-30.025459,-51.117782
304,way,249901840,clinic,NaN,NaN,NaN,-30.113241,-51.165271
334,way,288987786,school,NaN,NaN,NaN,-29.992983,-51.192296
335,way,288987796,doctors,NaN,NaN,NaN,-29.992663,-51.193657
372,way,317192910,school,NaN,NaN,NaN,-30.035335,-51.233530
388,way,318661243,hospital,NaN,NaN,NaN,-30.106660,-51.171792


### Pontos sem nome a menos de 200 m um do outro (possíveis duplicatas)

In [40]:
import numpy as np

LIMITE_M = 200

def haversine_m(lat1, lon1, lat2, lon2):
    """Distância em metros entre coordenadas (lat/lon em graus)."""
    R = 6_371_000
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    a = (np.sin((lat2 - lat1) / 2) ** 2
         + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2)
    return 2 * R * np.arcsin(np.sqrt(a))

s = sem_nome.reset_index(drop=True)
# Matriz de distâncias entre todos os pares de pontos sem nome
dist = haversine_m(s["lat"].values[:, None], s["lon"].values[:, None],
                   s["lat"].values[None, :], s["lon"].values[None, :])
i, j = np.where(np.triu(dist < LIMITE_M, k=1))  # cada par uma vez só

pares = pd.DataFrame({
    "id_1": s.loc[i, "id"].values, "tipo_1": s.loc[i, "type"].values,
    "amenity_1": s.loc[i, "tags.amenity"].values,
    "id_2": s.loc[j, "id"].values, "tipo_2": s.loc[j, "type"].values,
    "amenity_2": s.loc[j, "tags.amenity"].values,
    "distancia_m": dist[i, j].round(1),
}).sort_values("distancia_m")
print(f"{len(pares)} pares de pontos sem nome a menos de {LIMITE_M} m")
pares

12 pares de pontos sem nome a menos de 200 m


,id_1,tipo_1,amenity_1,id_2,tipo_2,amenity_2,distancia_m
11,425216986,way,school,425217265,way,school,10.7
6,425215193,way,school,425215214,way,school,11.5
0,8730543929,node,clinic,8730543930,node,clinic,18.2
8,425215193,way,school,425217265,way,school,24.2
10,425215214,way,school,425217265,way,school,27.6
7,425215193,way,school,425216986,way,school,34.5
4,425215043,way,school,425216986,way,school,35.1
5,425215043,way,school,425217265,way,school,36.5
9,425215214,way,school,425216986,way,school,38.2
3,425215043,way,school,425215214,way,school,48.7
